<a href="https://colab.research.google.com/github/MajidFQ/MultiModel-Spam-Classification/blob/main/spam_classifier_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request
import zipfile
import os

# Download the official SMS Spam dataset zip file
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam.zip"
urllib.request.urlretrieve(url, zip_path)

# Extract it
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("sms_data")

# Read it into a pandas DataFrame cleanly
df = pd.read_csv("sms_data/SMSSpamCollection", sep='\t', names=['label_raw', 'text'])

# Convert labels: 'ham' -> 0, 'spam' -> 1
df['label'] = df['label_raw'].map({'ham': 0, 'spam': 1})

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==========================================
# 1. DATA PREPARATION (Using Mock Data)
# ==========================================
# In Colab, replace this with: df = pd.read_csv('your_spam_dataset.csv')

# Vectorizing the text data
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['text'])
y = df['label']

# Splitting data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Master list to collect all final comparison metrics
table_rows = []

def record_metrics(model_name, variant, y_true, y_pred):
    """Helper function to calculate metrics and append to our master list."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    table_rows.append({
        "Model": model_name,
        "Variant/Parameter": variant,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Confusion Matrix": f"{cm[0][0]},{cm[0][1]}|{cm[1][0]},{cm[1][1]}"
    })

    print(f"\n--- {model_name} ({variant}) ---")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print("Confusion Matrix:\n", cm)

# ==========================================
# 2. MODEL 1: NAIVE BAYES (Custom Thresholds)
# ==========================================
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Get predicted probabilities for the positive class (Spam)
# index 0 is prob of Ham, index 1 is prob of Spam
y_probs_nb = nb_model.predict_proba(X_test)[:, 1]

thresholds = [0.3, 0.5, 0.75]
for t in thresholds:
    # Instead of predict(), manually flag as 1 if probability exceeds threshold
    y_pred_t = (y_probs_nb >= t).astype(int)
    record_metrics("Naive Bayes", f"Threshold: {t}", y_test, y_pred_t)

# ==========================================
# 3. MODEL 2: DECISION TREE (Varying Depths)
# ==========================================
depths = [25, 50, 75]
for d in depths:
    dt_model = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_model.fit(X_train, y_train)
    y_pred_dt = dt_model.predict(X_test)
    record_metrics("Decision Tree", f"Max Depth: {d}", y_test, y_pred_dt)

# ==========================================
# 4. MODEL 3: SVM (Varying Gamma)
# ==========================================
gammas = [0.01, 0.1, 0.5] # Changing hyperparameter gamma, avoiding 1.0
for g in gammas:
    # probability=True is needed if you ever want to check thresholds for SVM later
    svm_model = SVC(gamma=g, kernel='rbf', random_state=42)
    svm_model.fit(X_train, y_train)
    y_pred_svm = svm_model.predict(X_test)
    record_metrics("SVM", f"Gamma: {g}", y_test, y_pred_svm)

# ==========================================
# 5. FINAL COMPARISON TABLE
# ==========================================
print("\n" + "="*50 + "\nFINAL SUMMARY COMPARISON TABLE\n" + "="*50)
df_summary = pd.DataFrame(table_rows)
print(df_summary.to_string(index=False))


--- Naive Bayes (Threshold: 0.3) ---
Accuracy: 0.9839 | Precision: 0.9580 | Recall: 0.9195 | F1: 0.9384
Confusion Matrix:
 [[960   6]
 [ 12 137]]

--- Naive Bayes (Threshold: 0.5) ---
Accuracy: 0.9794 | Precision: 1.0000 | Recall: 0.8456 | F1: 0.9164
Confusion Matrix:
 [[966   0]
 [ 23 126]]

--- Naive Bayes (Threshold: 0.75) ---
Accuracy: 0.9453 | Precision: 1.0000 | Recall: 0.5906 | F1: 0.7426
Confusion Matrix:
 [[966   0]
 [ 61  88]]

--- Decision Tree (Max Depth: 25) ---
Accuracy: 0.9650 | Precision: 0.9167 | Recall: 0.8121 | F1: 0.8612
Confusion Matrix:
 [[955  11]
 [ 28 121]]

--- Decision Tree (Max Depth: 50) ---
Accuracy: 0.9695 | Precision: 0.9197 | Recall: 0.8456 | F1: 0.8811
Confusion Matrix:
 [[955  11]
 [ 23 126]]

--- Decision Tree (Max Depth: 75) ---
Accuracy: 0.9695 | Precision: 0.9197 | Recall: 0.8456 | F1: 0.8811
Confusion Matrix:
 [[955  11]
 [ 23 126]]

--- SVM (Gamma: 0.01) ---
Accuracy: 0.8664 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Confusion Matrix:
 [